# Treino do modelo: Árvore de Regressão para preço de imóveis

**Objetivo:** estimar o preço de venda de um imóvel em São Paulo a partir da **área**, do **número de quartos** e do **bairro**.

**Dataset:** *São Paulo Real Estate - Sale / Rent - April 2019* (Kaggle). Dados reais de anúncios, não simulados.

Etapas deste notebook:
1. Carregar e preparar os dados (funções do `preparacao.py`)
2. Montar o modelo: One-Hot Encoding do bairro + Árvore de Regressão
3. Escolher a profundidade da árvore com validação cruzada
4. Medir o erro nos dados de teste
5. Calcular a faixa de erro que o site vai mostrar
6. Salvar o modelo com joblib

## 1. Carregar e preparar os dados

A limpeza fica no `preparacao.py` para que o treino e o site usem exatamente as mesmas regras:
só vendas, **sem anúncios duplicados** (linhas idênticas em todas as colunas), sem ausentes, sem valores impossíveis e bairro sem o "/São Paulo".

Dois cuidados para o erro medido não ficar otimista:
- **Divisão por grupo:** imóveis com mesma área, quartos, bairro e preço (provavelmente unidades iguais do mesmo prédio) vão todos para o treino **ou** todos para o teste, nunca para os dois.
- **Extremos de preço por m²:** o 1% mais baixo e o 1% mais alto são removidos **depois** da divisão, com limites calculados só com o treino.

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

from preparacao import (
    carregar_dados, preparar_dados, dividir_treino_teste,
    grupos_imovel,
)

df = carregar_dados()
vendas = df[df["Negotiation Type"] == "sale"]
dados = preparar_dados(df)
X_treino, X_teste, y_treino, y_teste, (preco_m2_min, preco_m2_max) = dividir_treino_teste(dados)

print("Vendas no dataset:", len(vendas))
print("Duplicatas removidas:", len(vendas) - len(vendas.drop_duplicates()))
print("Imóveis após a preparação:", len(dados))
print(f"Preço/m² mantido entre R$ {preco_m2_min:,.0f} e R$ {preco_m2_max:,.0f}")
print("Treino:", len(X_treino), "| Teste:", len(X_teste))

Vendas no dataset: 6412
Duplicatas removidas: 110
Imóveis após a preparação: 6302
Preço/m² mantido entre R$ 2,500 e R$ 17,889
Treino: 4951 | Teste: 1224


## 2. Montar o modelo

O modelo é um **Pipeline**, uma "linha de montagem" com duas etapas:

1. **ColumnTransformer:** transforma o bairro (texto) em colunas de 0 ou 1 com **One-Hot Encoding**. Área e quartos passam direto (`passthrough`).
   - `handle_unknown="ignore"`: se chegar um bairro que o modelo nunca viu, ele não quebra (todas as colunas de bairro ficam 0).
   - **Não normalizamos** área e quartos: a árvore divide os dados por limiares ("área > 85?"), e mudar a escala não muda as divisões.
2. **DecisionTreeRegressor:** a Árvore de Regressão. Ela faz perguntas sobre os dados até chegar a um grupo de imóveis parecidos e devolve a **média de preço desse grupo**.

Com o Pipeline, o site manda `área, quartos, bairro` do jeito que o usuário digitou, e o próprio modelo faz a transformação.

In [2]:
transformador = ColumnTransformer(
    transformers=[
        ("bairro", OneHotEncoder(handle_unknown="ignore"), ["District"]),
    ],
    remainder="passthrough",  # Size e Rooms passam sem alteração
)

modelo = Pipeline([
    ("preparo", transformador),
    ("arvore", DecisionTreeRegressor(random_state=42)),
])
modelo

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preparo', ...), ('arvore', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('bairro', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output 

## 3. Escolher a profundidade da árvore (validação cruzada)

Uma árvore sem limite cresce até "decorar" o treino (**overfitting**) e erra muito em imóveis novos.
Uma árvore rasa demais é simples demais (**underfitting**). Dois parâmetros controlam isso:

- `max_depth`: quantas perguntas seguidas a árvore pode fazer.
- `min_samples_leaf`: quantos imóveis, no mínimo, cada grupo final precisa ter.

O **GridSearchCV** testa todas as combinações. Para cada uma, faz **validação cruzada com 5 partes** (`cv=5`):
divide o treino em 5, treina em 4 e mede o erro na 5ª, repetindo 5 vezes. Usamos **GroupKFold**: imóveis do mesmo grupo (mesma área, quartos, bairro e preço) ficam na mesma parte, como na divisão treino/teste.
**O teste não é usado aqui**, ele fica guardado para a avaliação final.

A métrica usada para escolher é o **MAE** (erro absoluto médio): em média, quantos reais a estimativa erra.

In [3]:
parametros = {
    "arvore__max_depth": [3, 5, 8, 10, 12, 15, 20, None],
    "arvore__min_samples_leaf": [1, 5, 10, 20],
}

busca = GridSearchCV(
    modelo,
    parametros,
    cv=GroupKFold(n_splits=5),
    scoring="neg_mean_absolute_error",  # o scikit-learn maximiza, por isso o MAE vem negativo
)
busca.fit(X_treino, y_treino, groups=grupos_imovel(X_treino, y_treino))

print("Melhores parâmetros:", busca.best_params_)
print(f"MAE médio na validação cruzada: R$ {-busca.best_score_:,.0f}")

Melhores parâmetros: {'arvore__max_depth': None, 'arvore__min_samples_leaf': 1}
MAE médio na validação cruzada: R$ 127,707


In [4]:
# Como o erro muda com a profundidade (com o melhor min_samples_leaf)
resultados = pd.DataFrame(busca.cv_results_)
melhor_folha = busca.best_params_["arvore__min_samples_leaf"]
tabela = resultados[resultados["param_arvore__min_samples_leaf"] == melhor_folha]
tabela = tabela[["param_arvore__max_depth", "mean_test_score"]].copy()
tabela["MAE (R$)"] = (-tabela["mean_test_score"]).round(0)
tabela[["param_arvore__max_depth", "MAE (R$)"]].rename(columns={"param_arvore__max_depth": "max_depth"})

,max_depth,MAE (R$)
0,3,162115.0
4,5,150519.0
8,8,141753.0
12,10,137941.0
16,12,135487.0
20,15,133640.0
24,20,129437.0
28,None,127707.0


## 4. Avaliar nos dados de teste

Agora usamos os imóveis que o modelo **nunca viu**. Três métricas:

- **MAE:** erro médio em reais. Fácil de explicar: "em média, erra R$ X".
- **RMSE:** parecido com o MAE, mas pune mais os erros grandes.
- **R²:** fração da variação do preço que o modelo explica (1 = perfeito, 0 = não explica nada).

Para saber se o modelo é bom, comparamos com um **modelo de referência** que sempre chuta a média de preço do treino.

In [5]:
melhor_modelo = busca.best_estimator_
previsto = melhor_modelo.predict(X_teste)

referencia = DummyRegressor(strategy="mean").fit(X_treino, y_treino)
previsto_ref = referencia.predict(X_teste)

def metricas(real, prev):
    return {
        "MAE (R$)": round(mean_absolute_error(real, prev)),
        "RMSE (R$)": round(np.sqrt(mean_squared_error(real, prev))),
        "R²": round(r2_score(real, prev), 3),
    }

pd.DataFrame({
    "Árvore de Regressão": metricas(y_teste, previsto),
    "Referência (sempre a média)": metricas(y_teste, previsto_ref),
})

,Árvore de Regressão,Referência (sempre a média)
MAE (R$),129418.000,346911.0
RMSE (R$),297131.000,554149.0
R²,0.712,-0.0


## 5. Faixa de erro para o site

Um erro de R$ 100 mil é pequeno para um imóvel de R$ 3 milhões e enorme para um de R$ 200 mil.
Por isso a faixa do site é **percentual**: medimos o erro relativo de cada imóvel do teste
e pegamos o valor que cobre **80% dos casos**.

Assim, o site pode dizer: *"Em 80% dos imóveis de teste, o preço real ficou dentro desta faixa."*

In [6]:
erro_relativo = np.abs(y_teste.to_numpy() - previsto) / previsto
faixa_pct = float(np.quantile(erro_relativo, 0.80))
erro_mediano_pct = float(np.median(erro_relativo))

print(f"Erro relativo mediano: {erro_mediano_pct:.1%}")
print(f"Faixa que cobre 80% dos casos: ±{faixa_pct:.1%}")

Erro relativo mediano: 12.8%
Faixa que cobre 80% dos casos: ±29.6%


### Quais variáveis a árvore mais usou?

A **importância** mostra quanto cada variável ajudou a reduzir o erro nas divisões da árvore.
As 96 colunas de bairro são somadas em uma só, para comparar com área e quartos.

In [7]:
nomes = melhor_modelo.named_steps["preparo"].get_feature_names_out()
importancias = pd.Series(melhor_modelo.named_steps["arvore"].feature_importances_, index=nomes)

agrupado = {
    "Área": importancias[[n for n in nomes if n.endswith("Size")]].sum(),
    "Quartos": importancias[[n for n in nomes if n.endswith("Rooms")]].sum(),
    "Bairro": importancias[[n for n in nomes if n.startswith("bairro__")]].sum(),
}
pd.Series(agrupado).sort_values(ascending=False).round(3)

Área       0.823
Bairro     0.157
Quartos    0.020
dtype: float64

### Teste com um imóvel de exemplo

In [8]:
exemplo = pd.DataFrame([{"Size": 80, "Rooms": 2, "District": "Pinheiros"}])
preco = melhor_modelo.predict(exemplo)[0]
print(f"Estimativa: R$ {preco:,.0f}")
print(f"Faixa: R$ {preco * (1 - faixa_pct):,.0f} a R$ {preco * (1 + faixa_pct):,.0f}")

Estimativa: R$ 694,500
Faixa: R$ 489,037 a R$ 899,963


## 6. Salvar o modelo

O site **não treina** a cada acesso: ele carrega o modelo pronto com `joblib`.
Junto, salvamos um arquivo de informações (`metadados.json`) com o que o site precisa para validar as entradas
e para a seção "Como funciona": lista de bairros, limites de área e quartos, métricas e faixa de erro.

In [9]:
pasta_modelo = Path("modelo")
pasta_modelo.mkdir(exist_ok=True)

joblib.dump(melhor_modelo, pasta_modelo / "modelo_arvore.joblib")

metricas_teste = metricas(y_teste, previsto)
metadados = {
    # limites e bairros vêm do treino: é o que o modelo realmente conhece
    "bairros": sorted(X_treino["District"].unique().tolist()),
    "area_min": int(X_treino["Size"].min()),
    "area_max": int(X_treino["Size"].max()),
    "quartos_min": int(X_treino["Rooms"].min()),
    "quartos_max": int(X_treino["Rooms"].max()),
    "preco_m2_min": round(preco_m2_min),
    "preco_m2_max": round(preco_m2_max),
    "faixa_pct": round(faixa_pct, 4),
    "erro_mediano_pct": round(erro_mediano_pct, 4),
    "mae": metricas_teste["MAE (R$)"],
    "rmse": metricas_teste["RMSE (R$)"],
    "r2": metricas_teste["R²"],
    "parametros": {k.replace("arvore__", ""): v for k, v in busca.best_params_.items()},
    "linhas_treino": len(X_treino),
    "linhas_teste": len(X_teste),
}
with open(pasta_modelo / "metadados.json", "w", encoding="utf-8") as f:
    json.dump(metadados, f, ensure_ascii=False, indent=2)

print("Modelo e metadados salvos em", pasta_modelo.resolve())

Modelo e metadados salvos em D:\mineração de dados projeto final\previsao-preco-imoveis\modelo
